# Modulo 2 - Clasificacion de conduccion distractiva

Este notebook entrena y evalua una linea base en PyTorch para clasificar comportamientos del conductor a partir de imagenes.

Dataset usado: [Multi-Class Driver Behavior Image Dataset](https://www.kaggle.com/datasets/arafatsahinafridi/multi-class-driver-behavior-image-dataset).

Clases esperadas:

- `Safe Driving`
- `Turning`
- `Texting Phone`
- `Talking Phones`
- `Others`

## 1. Preparacion del entorno

Ejecuta este notebook en Google Colab con GPU activada:

`Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU`

La instalacion se hace dentro del notebook para que el flujo sea reproducible en Colab.

In [ ]:
# Dependencias necesarias para entrenamiento, evaluacion y descarga desde Kaggle.
!pip install -q torch torchvision pandas scikit-learn matplotlib seaborn pillow kaggle


In [ ]:
# Imports generales del notebook.
from pathlib import Path
import os
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

# Semilla para hacer reproducible el split y el entrenamiento base.
torch.manual_seed(42)

# Colab usara CUDA si la GPU esta disponible.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dispositivo seleccionado: {DEVICE}")


## 2. Ubicacion del repositorio

Este notebook asume que el repositorio esta en `/content/rna-sistema_de_transporte_inteligente`.

Si lo clonaste en otra ruta, cambia `PROJECT_ROOT` antes de ejecutar la celda.

In [ ]:
PROJECT_ROOT = Path("/content/rna-sistema_de_transporte_inteligente")

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
else:
    raise FileNotFoundError(
        "No se encontro el repositorio. Clonalo o subelo en "
        f"{PROJECT_ROOT} antes de continuar."
    )

# Permite importar modulos locales como src.module2_distraction.
sys.path.insert(0, str(Path.cwd()))

print(f"Working directory: {Path.cwd()}")


## 3. Configuracion de Kaggle

Para descargar el dataset desde Kaggle necesitas credenciales.

Opcion recomendada en Colab:

1. En Kaggle, ve a `Account > API > Create New Token`.
2. Descarga `kaggle.json`.
3. Sube el archivo a Colab como `/content/kaggle.json`.

La siguiente celda copia ese archivo a la ruta esperada por la CLI de Kaggle.

In [ ]:
KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_JSON_COLAB = Path("/content/kaggle.json")
KAGGLE_JSON_TARGET = KAGGLE_DIR / "kaggle.json"

KAGGLE_DIR.mkdir(exist_ok=True)

if KAGGLE_JSON_COLAB.exists():
    shutil.copy(KAGGLE_JSON_COLAB, KAGGLE_JSON_TARGET)
    os.chmod(KAGGLE_JSON_TARGET, 0o600)
    print(f"Credenciales copiadas a {KAGGLE_JSON_TARGET}")
elif KAGGLE_JSON_TARGET.exists():
    print(f"Credenciales existentes en {KAGGLE_JSON_TARGET}")
else:
    raise FileNotFoundError(
        "No se encontro kaggle.json. Sube el archivo a /content/kaggle.json "
        "o configura KAGGLE_USERNAME y KAGGLE_KEY."
    )


## 4. Descarga del dataset

El script `scripts/download_data.py` descarga y extrae el dataset dentro de `data/raw/multi_class_driver_behavior`.

El loader del modulo busca automaticamente una carpeta compatible con `torchvision.datasets.ImageFolder`, incluso si Kaggle agrega una carpeta intermedia.

In [ ]:
DATA_ROOT = Path("data/raw/multi_class_driver_behavior")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Descarga oficial desde Kaggle. Puede tardar varios minutos porque el dataset pesa varios GB.
!python scripts/download_data.py --output-dir data/raw/multi_class_driver_behavior

print(f"Dataset disponible en: {DATA_ROOT.resolve()}")


## 5. Creacion de DataLoaders

Se crean splits estratificados por clase:

- `train`: entrenamiento
- `val`: seleccion del mejor checkpoint
- `test`: evaluacion final

Las transformaciones de entrenamiento incluyen aumentos moderados de encuadre, color y rotacion.

In [ ]:
from src.config import DEFAULT_BATCH_SIZE, DEFAULT_IMAGE_SIZE
from src.module2_distraction.data_loader import create_dataloaders

loaders, splits = create_dataloaders(
    data_root=DATA_ROOT,
    image_size=DEFAULT_IMAGE_SIZE,
    batch_size=DEFAULT_BATCH_SIZE,
    num_workers=2,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
    strict_classes=False,
)

# ImageFolder asigna indices alfabeticamente segun carpetas encontradas.
class_names = [splits.idx_to_class[index] for index in range(len(splits.idx_to_class))]

split_sizes = {
    "train": len(splits.train),
    "val": len(splits.val),
    "test": len(splits.test),
}

print("Clases detectadas:")
print(class_names)
print("\nTamano de splits:")
print(split_sizes)


## 6. Definicion del modelo

Se usa `resnet18` preentrenada en ImageNet como baseline. Es una arquitectura liviana y estable para Colab.

La cabeza final se reemplaza por una capa con salida igual al numero de clases detectadas.

In [ ]:
from src.module2_distraction.model import build_driver_behavior_model

model = build_driver_behavior_model(
    num_classes=len(class_names),
    model_name="resnet18",
    pretrained=True,
    freeze_backbone=False,
)

model = model.to(DEVICE)

print(model.__class__.__name__)
print(f"Numero de clases: {len(class_names)}")


## 7. Entrenamiento

El entrenamiento guarda el mejor checkpoint segun `F1 macro` de validacion.

Artefacto principal:

`models/distraction/best_model.pt`

In [ ]:
from src.module2_distraction.trainer import TrainingConfig, fit

training_config = TrainingConfig(
    epochs=12,
    learning_rate=1e-4,
    weight_decay=1e-4,
    patience=4,
    device=DEVICE,
    checkpoint_path="models/distraction/best_model.pt",
    model_name="resnet18",
    image_size=DEFAULT_IMAGE_SIZE,
    extra_metadata={
        "dataset": "arafatsahinafridi/multi-class-driver-behavior-image-dataset",
        "selection_metric": "val_f1_macro",
    },
)

training_summary = fit(
    model=model,
    dataloaders=loaders,
    class_names=class_names,
    class_to_idx=splits.class_to_idx,
    config=training_config,
)

training_summary


## 8. Evaluacion en test

Se calculan las metricas pedidas en el entregable:

- Accuracy
- Precision
- Recall
- F1-score
- Matriz de confusion
- Reporte por clase

In [ ]:
from src.module2_distraction.evaluator import (
    collect_predictions,
    distraction_frequency_report,
    evaluate_predictions,
    save_evaluation_artifacts,
)

ARTIFACT_DIR = Path("models/distraction/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Predicciones del conjunto de prueba.
y_true, y_pred, probabilities = collect_predictions(
    model=model,
    dataloader=loaders["test"],
    device=DEVICE,
)

# Metricas globales y por clase.
results = evaluate_predictions(
    y_true=y_true,
    y_pred=y_pred,
    class_names=class_names,
)

# Frecuencia de clases distractoras predichas.
results["distraction_frequency"] = distraction_frequency_report(
    y_pred=y_pred,
    class_names=class_names,
)

save_evaluation_artifacts(results, ARTIFACT_DIR)

pd.DataFrame(results["classification_report"])


## 9. Matriz de confusion

La matriz ayuda a identificar que clases se confunden entre si. Esto es importante para decidir medidas preventivas y mejoras del dataset.

In [ ]:
confusion_matrix = np.array(results["confusion_matrix"])

plt.figure(figsize=(8, 6))
sns.heatmap(
    confusion_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Prediccion")
plt.ylabel("Etiqueta real")
plt.title("Matriz de confusion - test")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "confusion_matrix.png", dpi=160)
plt.show()


## 10. Ejemplos correctos y casos erroneos

Estas grillas cumplen el entregable de ejemplos clasificados correctamente y errores. Cada imagen muestra etiqueta real, prediccion y confianza.

In [ ]:
from src.module2_distraction.augmentation import IMAGENET_MEAN, IMAGENET_STD


def denormalize_image(tensor):
    """Revierte la normalizacion ImageNet para visualizar una imagen."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    image = tensor.cpu() * std + mean
    return image.clamp(0, 1).permute(1, 2, 0).numpy()


def save_prediction_grid(indices, filename, title):
    """Guarda una grilla de hasta nueve ejemplos del conjunto de test."""
    if not indices:
        print(f"Sin ejemplos para: {title}")
        return

    selected_indices = indices[:9]
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    axes = axes.flatten()

    test_dataset = splits.test.dataset

    for axis, prediction_index in zip(axes, selected_indices):
        dataset_index = splits.test.indices[prediction_index]
        image, _ = test_dataset[dataset_index]

        true_name = class_names[y_true[prediction_index]]
        predicted_name = class_names[y_pred[prediction_index]]
        confidence = probabilities[prediction_index].max()

        axis.imshow(denormalize_image(image))
        axis.set_title(
            f"Real: {true_name}\nPred: {predicted_name} ({confidence:.2f})"
        )
        axis.axis("off")

    for axis in axes[len(selected_indices):]:
        axis.axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / filename, dpi=160)
    plt.show()


In [ ]:
correct_indices = np.where(y_true == y_pred)[0].tolist()
error_indices = np.where(y_true != y_pred)[0].tolist()

save_prediction_grid(
    indices=correct_indices,
    filename="correct_predictions.png",
    title="Ejemplos correctamente clasificados",
)

save_prediction_grid(
    indices=error_indices,
    filename="error_cases.png",
    title="Casos erroneos",
)


## 11. Distracciones frecuentes

La tabla ordena las clases distractoras por frecuencia predicha en test. Sirve como insumo para el informe preventivo.

In [ ]:
frequency_df = pd.DataFrame(results["distraction_frequency"])

frequency_df.to_csv(
    ARTIFACT_DIR / "distraction_frequency.csv",
    index=False,
)

frequency_df


## 12. Artefactos generados

Al terminar, revisa estos archivos:

- `models/distraction/best_model.pt`
- `models/distraction/training_history.json`
- `models/distraction/artifacts/metrics.json`
- `models/distraction/artifacts/classification_report.csv`
- `models/distraction/artifacts/confusion_matrix.csv`
- `models/distraction/artifacts/confusion_matrix.png`
- `models/distraction/artifacts/correct_predictions.png`
- `models/distraction/artifacts/error_cases.png`
- `models/distraction/artifacts/distraction_frequency.csv`

Con esos resultados se completa `docs/report/module2_distraction_report.md`.